# Codigo original que cuenta las veces que un usuario ha enviado un correo

In [21]:
import sqlite3

# Conectar con la base de datos.
# Si emaildb.sqlite no existe, SQLite la crea automáticamente.
conn = sqlite3.connect('emaildb.sqlite')

# Crear un cursor para ejecutar comandos SQL.
cur = conn.cursor()

# Eliminar la tabla Counts si ya existe.
# Esto permite empezar el conteo desde cero en cada ejecución.
cur.execute('DROP TABLE IF EXISTS Counts')

# Crear la tabla Counts con dos columnas:
# email: almacena el correo electrónico como texto.
# count: almacena el número de apariciones como entero.
cur.execute('''
CREATE TABLE Counts (email TEXT, count INTEGER)
''')

# Pedir al usuario el nombre del archivo que contiene los correos.
fname = input('Enter file name: ')

# Si el usuario no escribe ningún nombre, utilizar el archivo por defecto.
if len(fname) < 1:
    fname = 'mbox-short.txt'

# Abrir el archivo de texto para leerlo.
fh = open(fname)

# Recorrer el archivo línea por línea.
for line in fh:

    # Ignorar las líneas que no comienzan con 'From '.
    # Estas líneas no se utilizan para contar los correos.
    if not line.startswith('From: '):
        continue

    # Separar la línea en palabras usando los espacios.
    pieces = line.split()

    # El correo electrónico está en la segunda posición (índice 1).
    email = pieces[1]

    # Buscar si ese correo ya existe en la tabla Counts.
    # El signo ? es un marcador para pasar el valor de forma segura.
    cur.execute(
        'SELECT count FROM Counts WHERE email = ?',
        (email,)
    )

    # Obtener el primer resultado de la consulta.
    row = cur.fetchone()

    # Si no existe ningún registro para ese correo...
    if row is None:

        # Insertar el correo por primera vez, con un conteo de 1.
        cur.execute('''
            INSERT INTO Counts (email, count)
            VALUES (?, 1)
        ''', (email,))

    else:

        # Si el correo ya existe, incrementar su contador en 1.
        cur.execute('''
            UPDATE Counts
            SET count = count + 1
            WHERE email = ?
        ''', (email,))
        
# Guardar los cambios realizados en la base de datos.
conn.commit()

# Definir una consulta para obtener los 10 correos más frecuentes.
# ORDER BY count DESC ordena de mayor a menor frecuencia.
# LIMIT 10 devuelve como máximo 10 registros.
sqlstr = '''
SELECT email, count
FROM Counts
ORDER BY count DESC
LIMIT 10
'''

# Ejecutar la consulta y recorrer los resultados.
for row in cur.execute(sqlstr):

    # Mostrar el correo y su número de apariciones.
    print(str(row[0]), row[1])

# Cerrar el archivo y la conexión a la base de datos.
fh.close()
conn.close()

Enter file name: 
zqian@umich.edu 195
mmmay@indiana.edu 161
cwen@iupui.edu 158
chmaurer@iupui.edu 111
aaronz@vt.edu 110
ian@caret.cam.ac.uk 96
jimeng@umich.edu 93
rjlowe@iupui.edu 90
dlhaines@umich.edu 84
david.horwitz@uct.ac.za 67
